# 面试问题：Greenlist Watermark 怎样嵌入和检测，为什么不能把检测分数当作绝对证据？

可以直接复述的回答是：第一，用秘密 key 和前一个 token 为当前位置生成伪随机 greenlist。第二，生成时提高 green token 概率，但不强制固定词。第三，检测时重建每个位置的 greenlist，统计命中数并计算 z-score。第四，短文本、主题偏差和改写会影响统计功效。第五，应设置最小长度和 detected/uncertain/not_detected 三态。第六，水印只是一条来源信号，还需签名、日志和内容凭证。下面用五条客服内容生成任务演示。

## 真实案例：内容平台标记五类自动生成客服文本

五条请求覆盖退款、物流、账户、合同和故障通知。教学生成器从 24 个中文 token 中采样 48 token 文本，并以固定教学 key 控制 greenlist；输出仅为机制示例，不代表自然语言质量。生产 key 必须保密且轮换，不能像本 Notebook 一样明文保存。

In [1]:
import hashlib  # 使用 SHA-256 从上下文和 key 派生 greenlist 种子
import math  # 使用正态近似计算水印 z-score
import random  # 使用局部随机数生成确定性 token 序列
requests = [  # 定义五类自动内容生成任务
    {"id": "WM-01", "topic": "退款进度", "seed_token": "退款"},  # 退款客服文本
    {"id": "WM-02", "topic": "物流延迟", "seed_token": "物流"},  # 物流解释文本
    {"id": "WM-03", "topic": "账户安全", "seed_token": "账户"},  # 安全提醒文本
    {"id": "WM-04", "topic": "合同摘要", "seed_token": "合同"},  # 合同归纳文本
    {"id": "WM-05", "topic": "故障通知", "seed_token": "故障"},  # 服务状态文本
]  # 结束五个生成请求
vocabulary = ["退款", "物流", "账户", "合同", "故障", "已", "正在", "处理", "预计", "用户", "服务", "订单", "安全", "确认", "时间", "请", "稍后", "完成", "通知", "查询", "支持", "信息", "系统", "恢复"]  # 定义可读教学词表
gamma = 0.5  # 让每个位置一半词表属于 greenlist
teaching_key = "demo-key-2026"  # 定义仅供教学复现的非生产密钥
print("生成输入：id | topic | seed_token")  # 展示水印生成器接收的业务任务
for request in requests:  # 逐条输出五类请求
    print(f"{request['id']} | {request['topic']} | {request['seed_token']}")  # 呈现主题与首 token 上下文
print(f"词表大小={len(vocabulary)}，greenlist 比例={gamma:.0%}，每条长度=48")  # 展示统计检测基本参数


生成输入：id | topic | seed_token
WM-01 | 退款进度 | 退款
WM-02 | 物流延迟 | 物流
WM-03 | 账户安全 | 账户
WM-04 | 合同摘要 | 合同
WM-05 | 故障通知 | 故障
词表大小=24，greenlist 比例=50%，每条长度=48


## Baseline / 基线：静态绿色词表容易受主题偏差影响

若把“系统、服务、处理”等常见词固定为绿色，客服主题本身就可能产生高命中，导致普通文本误报。

In [2]:
static_green = {"系统", "服务", "处理", "用户", "信息", "通知"}  # 定义不依赖上下文的脆弱静态绿色词表
plain_topic_text = "系统 服务 正在 处理 用户 信息 系统 服务 完成 通知".split()  # 构造自然包含常见客服词的普通文本
static_hits = sum(token in static_green for token in plain_topic_text)  # 统计静态词表命中数
static_rate = static_hits / len(plain_topic_text)  # 计算普通主题文本的静态绿色比例
static_false_positive = static_rate > 0.5  # 使用简单阈值产生误报
print("普通客服文本：", " ".join(plain_topic_text))  # 展示没有嵌入水印的主题文本
print(f"静态绿色命中={static_hits}/{len(plain_topic_text)}，误报={static_false_positive}")  # 展示主题偏差问题


普通客服文本： 系统 服务 正在 处理 用户 信息 系统 服务 完成 通知
静态绿色命中=8/10，误报=True


## 核心实现：上下文 Greenlist 生成与 z-score 检测

每个位置的 greenlist 由 key、前一 token 和位置共同决定。水印生成器以 85% 概率从 greenlist 采样；普通生成器从完整词表均匀采样。

In [3]:
def greenlist(previous_token, position, key=teaching_key):  # 为单个位置生成伪随机绿色词集合
    payload = f"{key}|{previous_token}|{position}".encode("utf-8")  # 绑定密钥、上下文 token 和位置
    seed = int.from_bytes(hashlib.sha256(payload).digest()[:8], "big")  # 从摘要前八字节派生稳定整数种子
    local = random.Random(seed)  # 创建不影响全局状态的局部随机数生成器
    shuffled = list(vocabulary)  # 复制词表避免原地修改全局顺序
    local.shuffle(shuffled)  # 按上下文确定性打乱词表
    return set(shuffled[:int(len(vocabulary) * gamma)])  # 取固定比例 token 组成 greenlist
def generate(seed_token, watermarked, length=48, seed=0):  # 生成普通或带水印的教学 token 序列
    local = random.Random(seed)  # 使用请求级随机种子保证复现
    tokens = [seed_token]  # 以业务主题 token 作为首个上下文
    for position in range(1, length):  # 从第二个位置开始逐 token 生成
        greens = sorted(greenlist(tokens[-1], position))  # 重建当前位置的上下文绿色集合
        if watermarked and local.random() < 0.85:  # 带水印生成大多数位置偏向 greenlist
            token = local.choice(greens)  # 从绿色候选中随机选择一个词
        else:  # 普通生成或少量红色探索使用完整词表
            token = local.choice(vocabulary)  # 从全词表均匀采样
        tokens.append(token)  # 把当前 token 加入序列供下一位置建表
    return tokens  # 返回完整可检测 token 序列
def detect(tokens, key=teaching_key):  # 重建上下文 greenlist 并计算统计显著性
    hits = 0  # 初始化绿色命中数
    for position in range(1, len(tokens)):  # 首 token 没有前文因此不参与检测
        hits += int(tokens[position] in greenlist(tokens[position - 1], position, key))  # 逐位置验证绿色成员关系
    trials = len(tokens) - 1  # 记录参与统计的位置数
    expected = gamma * trials  # 计算无水印假设下期望命中数
    deviation = math.sqrt(trials * gamma * (1 - gamma))  # 计算二项分布标准差
    z_score = (hits - expected) / deviation if deviation else 0.0  # 计算标准化绿色偏移
    decision = "uncertain" if trials < 20 else ("detected" if z_score >= 3.0 else ("uncertain" if z_score >= 1.5 else "not_detected"))  # 使用最小长度和三态阈值
    return hits, trials, z_score, decision  # 返回可审计统计量
watermarked_outputs = {request["id"]: generate(request["seed_token"], True, seed=2100 + index) for index, request in enumerate(requests)}  # 为五条任务生成带水印文本
plain_outputs = {request["id"]: generate(request["seed_token"], False, seed=3100 + index) for index, request in enumerate(requests)}  # 为同五类任务生成普通文本
focus_tokens = watermarked_outputs["WM-01"]  # 选择退款文本展示位置检测
focus_detection = detect(focus_tokens)  # 计算退款文本命中数与 z-score
print("WM-01 前 20 token：", " ".join(focus_tokens[:20]))  # 展示实际生成的可读 token 序列
print(f"WM-01 检测：hits={focus_detection[0]}/{focus_detection[1]}，z={focus_detection[2]:.2f}，decision={focus_detection[3]}")  # 展示核心统计过程


WM-01 前 20 token： 退款 稍后 时间 稍后 订单 处理 正在 确认 预计 信息 信息 请 通知 服务 系统 支持 确认 服务 通知 时间
WM-01 检测：hits=40/47，z=4.81，decision=detected


## 失败案例与修正：短文本与大量改写降低统计功效

短文本即使全部命中也样本不足；长文本被随机替换三成 token 后，z-score 可能下降。检测器应输出 uncertain 而不是强行二分类，并结合签名来源。

In [4]:
short_tokens = focus_tokens[:10]  # 截取十 token 模拟标题或短信
short_detection = detect(short_tokens)  # 在不足二十个试验位置时执行检测
tampered_tokens = list(focus_tokens)  # 复制带水印文本以模拟人工改写
tamper_rng = random.Random(2721)  # 固定改写位置和替换 token
tampered_positions = sorted(tamper_rng.sample(range(1, len(tampered_tokens)), 15))  # 随机选择约三成位置
for position in tampered_positions:  # 逐位置替换原水印 token
    tampered_tokens[position] = tamper_rng.choice(vocabulary)  # 使用普通词表模拟同义改写或删改
tampered_detection = detect(tampered_tokens)  # 计算改写后的绿色命中统计
print(f"短文本：length={len(short_tokens)}，z={short_detection[2]:.2f}，decision={short_detection[3]}")  # 展示最小长度门禁
print(f"改写前：z={focus_detection[2]:.2f}，改写后：z={tampered_detection[2]:.2f}，替换位置={tampered_positions}")  # 展示鲁棒性下降
print("修正策略：uncertain 内容进入签名日志与来源凭证复核")  # 明确水印不能单独定罪


短文本：length=10，z=1.67，decision=uncertain
改写前：z=4.81，改写后：z=2.19，替换位置=[3, 4, 6, 11, 13, 15, 16, 23, 24, 25, 31, 35, 37, 40, 46]
修正策略：uncertain 内容进入签名日志与来源凭证复核


## 结果表：五类内容的水印与普通文本对照

In [5]:
watermarked_z = []  # 收集五条带水印文本的 z-score
plain_z = []  # 收集五条普通文本的 z-score
print("id | wm_hits | wm_z | wm_decision | plain_hits | plain_z | plain_decision")  # 输出逐请求检测对照
for request in requests:  # 使用相同检测器比较两种生成过程
    wm_result = detect(watermarked_outputs[request["id"]])  # 检测带水印文本
    plain_result = detect(plain_outputs[request["id"]])  # 检测普通文本
    watermarked_z.append(wm_result[2])  # 保存带水印显著性
    plain_z.append(plain_result[2])  # 保存普通文本显著性
    print(f"{request['id']} | {wm_result[0]:2}/{wm_result[1]} | {wm_result[2]:5.2f} | {wm_result[3]:12} | {plain_result[0]:2}/{plain_result[1]} | {plain_result[2]:5.2f} | {plain_result[3]}")  # 展示命中数、z 与三态决定
mean_watermarked_z = sum(watermarked_z) / len(watermarked_z)  # 计算五条带水印平均 z
mean_plain_z = sum(plain_z) / len(plain_z)  # 计算五条普通文本平均 z
detected_count = sum(detect(tokens)[3] == "detected" for tokens in watermarked_outputs.values())  # 统计成功检测的水印文本数
print(f"平均 z：watermarked={mean_watermarked_z:.2f}，plain={mean_plain_z:.2f}；detected={detected_count}/5")  # 输出整体统计分离度


id | wm_hits | wm_z | wm_decision | plain_hits | plain_z | plain_decision
WM-01 | 40/47 |  4.81 | detected     | 21/47 | -0.73 | not_detected
WM-02 | 42/47 |  5.40 | detected     | 26/47 |  0.73 | not_detected
WM-03 | 44/47 |  5.98 | detected     | 20/47 | -1.02 | not_detected
WM-04 | 46/47 |  6.56 | detected     | 27/47 |  1.02 | not_detected
WM-05 | 42/47 |  5.40 | detected     | 30/47 |  1.90 | uncertain
平均 z：watermarked=5.63，plain=0.38；detected=5/5


## 结果解读

上下文 greenlist 让每个位置的绿色集合变化，普通客服主题无法仅凭常见词稳定命中。五条带水印输出的平均 z 明显高于普通输出。短文本被标成 uncertain，改写后的分数下降，说明水印检测是概率证据而非内容所有权的绝对证明。

## 生产边界

真实实现需在模型 logits 上施加绿色偏置，评估困惑度、语言质量、攻击鲁棒性、多 key 轮换和多重检验。Tokenizer、温度和采样策略都会改变检测分布。密钥不能写进模型卡或客户端，检测结论应与 C2PA、服务日志或数字签名组合。本例词表很小且输出不自然。

## 最小回归测试

In [6]:
assert len(requests) >= 5  # 保证水印评测覆盖至少五类真实生成任务
assert static_false_positive is True  # 保证静态绿色词表基线真实暴露主题误报
assert focus_detection[3] == "detected"  # 保证核心退款文本达到教学检测阈值
assert short_detection[3] == "uncertain"  # 保证短文本不会被强行判定来源
assert mean_watermarked_z > mean_plain_z + 2.0  # 保证带水印与普通文本在五条样本上显著分离
assert detected_count >= 4  # 保证大多数带水印教学文本被检测
